In [1]:
import os

data_dir = "/mnt/d/deep_learning/datasets/mnist"

os.listdir(data_dir)

['t10k-images-idx3-ubyte.gz',
 't10k-labels-idx1-ubyte.gz',
 'train-images-idx3-ubyte.gz',
 'train-labels-idx1-ubyte.gz']

In [2]:
import gzip
import struct
import numpy as np


def load_mnist_images(path):
    with gzip.open(path, "rb") as f:
        magic, num, rows, cols = struct.unpack(">IIII", f.read(16))
        if magic != 2051:
            raise ValueError(f"Unexpected magic number for images: {magic}")
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.reshape(num, rows, cols)


def load_mnist_labels(path):
    with gzip.open(path, "rb") as f:
        magic, num = struct.unpack(">II", f.read(8))
        if magic != 2049:
            raise ValueError(f"Unexpected magic number for labels: {magic}")
        labels = np.frombuffer(f.read(), dtype=np.uint8)
    return labels


train_images = load_mnist_images(os.path.join(data_dir, "train-images-idx3-ubyte.gz"))
train_labels = load_mnist_labels(os.path.join(data_dir, "train-labels-idx1-ubyte.gz"))
test_images = load_mnist_images(os.path.join(data_dir, "t10k-images-idx3-ubyte.gz"))
test_labels = load_mnist_labels(os.path.join(data_dir, "t10k-labels-idx1-ubyte.gz"))

print(train_images.shape, train_labels.shape)
print(test_images.shape, test_labels.shape)


(60000, 28, 28) (60000,)
(10000, 28, 28) (10000,)


In [3]:
import torch
from torch.utils.data import Dataset, DataLoader

class MNISTDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = torch.from_numpy(images.astype(np.float32) / 255.0)
        self.labels = torch.from_numpy(labels.astype(np.int64))
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = self.images[index]
        label = self.labels[index]

        if self.transform is not None:
            image = self.transform(image)

        return image, label


train_dataset = MNISTDataset(train_images, train_labels)
test_dataset = MNISTDataset(test_images, test_labels)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(len(train_loader), len(test_loader))
print(next(iter(train_loader))[0].shape, next(iter(train_loader))[1].shape)


938 157
torch.Size([64, 28, 28]) torch.Size([64])


In [4]:
import torch.nn as nn


class MLP(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=128, output_dim=10):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.network(x)


model = MLP()

batch_images, _ = next(iter(train_loader))
out = model(batch_images)

print(model)
print(out.shape)


MLP(
  (network): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=10, bias=True)
  )
)
torch.Size([64, 10])


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, targets in train_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += targets.size(0)
        correct += (predicted == targets).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct / total
    print(f"Epoch {epoch+1:2d}/{num_epochs} - loss: {epoch_loss:.4f} - acc: {epoch_acc:.4f}")


Epoch  1/10 - loss: 0.3412 - acc: 0.9083
Epoch  2/10 - loss: 0.1543 - acc: 0.9554
Epoch  3/10 - loss: 0.1077 - acc: 0.9689
Epoch  4/10 - loss: 0.0810 - acc: 0.9756
Epoch  5/10 - loss: 0.0647 - acc: 0.9809
Epoch  6/10 - loss: 0.0523 - acc: 0.9843
Epoch  7/10 - loss: 0.0428 - acc: 0.9869
Epoch  8/10 - loss: 0.0358 - acc: 0.9893
Epoch  9/10 - loss: 0.0301 - acc: 0.9910
Epoch 10/10 - loss: 0.0247 - acc: 0.9930


In [6]:
model.eval()
with torch.no_grad():
    test_correct = 0
    test_total = 0
    for inputs, targets in test_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)

        test_total += targets.size(0)
        test_correct += (predicted == targets).sum().item()

print(f"Test accuracy: {test_correct / test_total:.4f}")

Test accuracy: 0.9781
